# 3 — Embeddings

**This is the experiment the project is built around.**

A model cannot read words, only numbers. An *embedding* is a table that maps each word to a
vector of 300 numbers, arranged so that words used in similar ways end up near each other.

The question this project asks is:

> **Do word vectors trained on medical text beat general-purpose English word vectors?**

To answer it cleanly, four embedding tables are built. Later notebooks train the *exact
same model* four times, changing nothing but which table it starts from.

| | What it is | Trained on |
|---|---|---|
| **E0** | random vectors | nothing — the floor |
| **E1** | GloVe 6B, 300d | Wikipedia + Gigaword (general English) |
| **E2** | Word2Vec, 300d | **this project's 159,975 PubMed abstracts** |
| **E3** | FastText, 300d | the same PubMed abstracts |

E0 is the control: a model that knows nothing about words at all. Whatever E1/E2/E3 score
above E0 is what the vectors contributed.

In [1]:
import sys, json, textwrap
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_colwidth", 90)
print("project root:", ROOT)

project root: E:\CSE\NLP Project


---

## 3.1 Training E2 and E3

Both are trained with `gensim` on `data/pubmed_sentences.txt` (1.88M sentences, 40.5M
words, from notebook 2). They share every hyperparameter, so the only difference between
them is the algorithm.

Two choices worth being able to defend:

- **`sg=1` (skip-gram) rather than CBOW.** Skip-gram is better on rare words, and the rare
  words here are the drug names the whole project is about.
- **The difference between Word2Vec and FastText** is that FastText represents a word as
  the sum of its character n-grams. So it can build a vector for a word it has never seen,
  by assembling it from pieces. That turns out to matter — and not entirely in the way you
  would expect.

E1 is not trained at all, just downloaded.

> **The code below is real**, and it is what produced `models/w2v.kv` and `models/ft.kv`.
> It sits behind a `TRAIN` switch that is **off** by default: training both models takes
> about an hour and needs the 270 MB sentence file. The finished vectors are already in
> `models/`, and the rest of this notebook loads and inspects them.

In [2]:
# ---------------------------------------------------------------------------
# OFF by default. Training E2 and E3 takes ~1 hour and needs
# data/pubmed_sentences.txt (270 MB, built in notebook 2).
# ---------------------------------------------------------------------------
TRAIN = False

MODELS = ROOT / "models"
SENTENCES = ROOT / "data" / "pubmed_sentences.txt"

# PRD section 7.1. Shared by both models, deliberately: the only difference
# between E2 and E3 must be the algorithm.
PARAMS = dict(
    vector_size=300,   # matches GloVe, so E1/E2/E3 are dimensionally comparable
    window=5,          # how many words either side count as "context"
    min_count=5,       # a word seen 4 times or fewer gets no vector
    negative=10,       # negative sampling
    epochs=5,
    sg=1,              # skip-gram, not CBOW - better on the rare drug names
    sample=1e-4,       # downsample very frequent words
    seed=42,
)


def train_embedding(kind, workers=None):
    """Train E2 (`w2v`) or E3 (`ft`) on the PubMed sentences."""
    import multiprocessing as mp
    import time

    from gensim.models import FastText, Word2Vec
    from gensim.models.word2vec import LineSentence

    if not SENTENCES.exists():
        raise FileNotFoundError(f"{SENTENCES} not found - build it in notebook 2 first.")

    workers = workers or max(mp.cpu_count() - 1, 1)
    builder = {"w2v": Word2Vec, "ft": FastText}[kind]

    print(f"training {kind} on {SENTENCES.name} with {workers} workers")
    t0 = time.time()
    model = builder(corpus_file=str(SENTENCES), workers=workers, **PARAMS)
    minutes = (time.time() - t0) / 60

    # Save only the vectors, not the training state: the full model carries the
    # negative-sampling tables and is several times larger for no later use.
    out = MODELS / f"{kind}.kv"
    model.wv.save(str(out))
    print(f"  {len(model.wv.index_to_key):,} words x {model.wv.vector_size}d "
          f"in {minutes:.1f} min -> {out.name}")
    return out


def download_glove():
    """E1: the general-purpose baseline, downloaded rather than trained."""
    import gensim.downloader

    kv = gensim.downloader.load("glove-wiki-gigaword-300")
    kv.save(str(MODELS / "glove.kv"))
    print(f"  GloVe {len(kv.index_to_key):,} words -> glove.kv")


if TRAIN:
    download_glove()
    train_embedding("w2v")      # E2
    train_embedding("ft")       # E3

In [3]:
import gc, time
from src.embeddings import FILES, load, nearest, covers

# Loaded one at a time on purpose: GloVe alone is 458 MB of vectors, and holding
# all three at once will run a laptop out of memory. `src/embeddings.py` also
# memory-maps them and does the similarity search in blocks, for the same reason.
def for_each_embedding(fn):
    """Load each embedding, apply `fn` to it, free it again. Returns {name: result}."""
    results = {}
    for name in FILES:
        t0 = time.perf_counter()
        kv = load(name)
        results[name] = fn(kv)
        print(f"  {name:20s} {len(kv.index_to_key):>7,} words x {kv.vector_size}d "
              f"({time.perf_counter() - t0:.1f}s)")
        del kv
        gc.collect()
    return results

_ = for_each_embedding(lambda kv: len(kv.index_to_key))

  E1 GloVe (general)   400,000 words x 300d (1.4s)
  E2 Word2Vec (ours)   113,103 words x 300d (0.1s)


  E3 FastText (ours)   113,103 words x 300d (16.4s)


---

## 3.2 What the three embeddings actually learned

The clearest way to see the difference is to ask each one: *which words are most similar to
this one?* Nothing about the medical domain is built into that question — the answer comes
purely from how the words were used in the training text.

In [4]:
PROBES = ["hepatotoxicity", "doxorubicin", "nephrotoxicity", "withdrawal", "induced"]

def neighbours(kv, word, k=5):
    hits = nearest(kv, word, k)
    if not hits:
        return "** not in this vocabulary **"
    return ", ".join(f"{w} ({score:.2f})" for w, score in hits)

found = for_each_embedding(lambda kv: {p: neighbours(kv, p) for p in PROBES})

for probe in PROBES:
    print()
    print(f"### {probe}")
    for name in FILES:
        print(f"  {name:20s} {found[name][probe]}")

  E1 GloVe (general)   400,000 words x 300d (1.0s)


  E2 Word2Vec (ours)   113,103 words x 300d (0.3s)


  E3 FastText (ours)   113,103 words x 300d (14.3s)

### hepatotoxicity
  E1 GloVe (general)   side-effects (0.52), dyskinesia (0.50), agranulocytosis (0.47), tardive (0.47), neurotoxicity (0.47)
  E2 Word2Vec (ours)   nephrotoxicity (0.69), hepatoxicity (0.68), hepatotoxic (0.65), cardiotoxicity (0.64), n-acetyl-p-aminophenol (0.63)
  E3 FastText (ours)   hepatotoxicities (0.94), hepatoxicity (0.93), hepatotoxic (0.91), hepatotoxicant (0.87), hepatonephrotoxicity (0.87)

### doxorubicin
  E1 GloVe (general)   cisplatin (0.65), cyclophosphamide (0.63), chemotherapeutic (0.56), vincristine (0.56), etoposide (0.56)
  E2 Word2Vec (ours)   dox (0.71), adriamycin (0.71), anthracycline (0.65), mitoxantrone (0.64), naca (0.63)
  E3 FastText (ours)   doxorubicine (0.96), doxorubicinol (0.94), doxorubicin-based (0.88), amrubicin (0.87), daunorubicin (0.87)

### nephrotoxicity
  E1 GloVe (general)   mangxamba (0.51), zety (0.49), mongkolporn (0.48), ______________________________________________

Three different kinds of answer:

**E1 GloVe returns loosely topical words.** For `hepatotoxicity` it offers `side-effects`,
`tardive`, `dyskinesia` — medical-*sounding*, but not the same concept. And for
`nephrotoxicity` it returns nonsense: names and punctuation runs scraped from the web.
GloVe has seen the word, but too rarely to have learned anything about it.

**E2 Word2Vec returns semantic relatives** — which is what a distributional model is
supposed to do:

| Probe | E2's neighbours | The relationship |
|---|---|---|
| `doxorubicin` | adriamycin, anthracycline, mitoxantrone | brand synonym, drug class, sibling drug |
| `hepatotoxicity` | nephrotoxicity, cardiotoxicity | sibling organ toxicities |
| `withdrawal` | abstinence, naloxone-precipitated | clinically co-occurring concepts |

**E3 FastText returns morphological variants**, and this was not what anyone expected:

| Probe | E3's neighbours | The relationship |
|---|---|---|
| `doxorubicin` | doxorubicine, doxorubicinol, doxorubicin-based | spellings and derivations |
| `hepatotoxicity` | hepatotoxicities, hepatoxicity, hepatotoxic | inflections, and a *misspelling* |
| `withdrawal` | withdrawals, withdraw, **withdrawalpolicy** | string overlap, including junk |

That is the character-n-gram model doing exactly what it is built to do: measuring string
similarity. Note it is useful *and* noisy — `hepatoxicity` is a genuine misspelling
correctly matched to the right concept, while `withdrawalpolicy` is a tokenisation
accident that only looks similar.

Keep this in mind for notebook 4: E3 goes on to score highest, but the mechanism is
morphological robustness, not deeper semantic knowledge.

---

## 3.3 Coverage: how many of our words does each embedding even know?

Before asking which vectors are better, there is a blunter question: does the embedding
have a vector for our words at all? Computed live over all 20,896 corpus sentences.

In [5]:
from collections import Counter
from src.tokenizer import tokenize
from src.vocab import is_indexable

corpus = pd.concat([pd.read_parquet(ROOT / "data" / "splits" / f"stage1_{s}.parquet")
                    for s in ["train", "dev", "test"]])["text"]

counts = Counter()
for sentence in corpus:
    counts.update(w for w in tokenize(sentence)[0] if is_indexable(w))

total_tokens = sum(counts.values())
print(f"{len(corpus):,} sentences, {total_tokens:,} words, {len(counts):,} distinct")
print()

def coverage(kv):
    known = [w for w in counts if covers(kv, w)]
    return {"type coverage": f"{len(known) / len(counts):.2%}",
            "token coverage": f"{sum(counts[w] for w in known) / total_tokens:.2%}",
            "words known": f"{len(known):,}",
            "words missing": f"{len(counts) - len(known):,}"}

pd.DataFrame(for_each_embedding(coverage)).T.rename_axis("embedding").reset_index()

20,896 sentences, 373,201 words, 20,298 distinct

  E1 GloVe (general)   400,000 words x 300d (0.1s)


  E2 Word2Vec (ours)   113,103 words x 300d (0.1s)


  E3 FastText (ours)   113,103 words x 300d (18.4s)


,embedding,type coverage,token coverage,words known,words missing
0,E1 GloVe (general),66.18%,95.67%,"13,434","6,864"
1,E2 Word2Vec (ours),79.86%,98.31%,"16,210","4,088"
2,E3 FastText (ours),100.00%,100.00%,"20,298",0


### How to read this table

**Token coverage is high for everyone; type coverage is not. That gap is the finding.**

General-purpose vectors cover the common English scaffolding — `the`, `patient`, `was`,
`after` — and that is most of the running text, so token coverage flatters them. Type
coverage counts each distinct word once, and it is the distinct words that carry the
domain. A drug name appearing three times is as informative as one appearing three hundred.

GloVe is missing **6,864 distinct words** the corpus uses. Some of the most frequent:

| Missing from GloVe | Times it appears |
|---|---|
| `patient's` | 193 |
| `intravitreal` | 75 |
| `crohn's` | 68 |
| `endophthalmitis` | 55 |
| `rechallenge` | 46 |
| `siadh` | 36 |

**E3's 100% is by construction, not by merit.** FastText builds a vector for *any* string
from character n-grams, so membership is always true. This metric literally cannot
distinguish it from a model that genuinely knows the word. Its real claim is that it has no
out-of-vocabulary words *by design*; whether those synthesised vectors are any good is what
the nearest-neighbour table above and the F1 comparison in notebook 4 actually test.

---

## 3.4 Building the four matrices — where the experiment is made fair

The models do not use these embeddings directly. Each one is projected onto **one shared
vocabulary**, producing four `.npy` files of identical shape.

This step is what makes the comparison valid. If the four runs used different word lists,
a difference in F1 could be a difference in *which words exist* rather than in what the
vectors know. Here the word list is fixed and only the numbers change.

**Why one shared random base matters.** Every matrix starts from the *same* seeded draw,
and each embedding overwrites only the rows it covers. So the rows nobody covers are
byte-identical across all four files. Drawing fresh noise per matrix would mean E1's
uncovered third and E2's uncovered fifth held *different* random values — and part of any
measured difference would be that noise rather than the embeddings.

This one also runs on a laptop in about a minute, but it is gated too, because it
overwrites the matrices every trained model in the project was built against.

In [6]:
def build_embedding_matrices(min_freq=2, dim=300, seed=42):
    """Project E1/E2/E3 onto one shared vocabulary. Writes models/emb_matrices/."""
    import gc

    import numpy as np

    from src.embeddings import FILES, build_matrix, load, task_token_counts

    PAD, UNK = "<pad>", "<unk>"
    out_dir = ROOT / "models" / "emb_matrices"
    out_dir.mkdir(parents=True, exist_ok=True)

    texts = pd.concat([pd.read_parquet(ROOT / "data" / "splits" / f"stage1_{s}.parquet")
                       for s in ("train", "dev", "test")])["text"].tolist()
    counts = task_token_counts(texts)

    # Sorted, so the row order is a function of the vocabulary alone and does
    # not depend on dict iteration order.
    vocab = [PAD, UNK] + sorted(t for t, c in counts.items() if c >= min_freq)
    index = {t: i for i, t in enumerate(vocab)}
    dropped = len(counts) - (len(vocab) - 2)

    print(f"{len(texts):,} sentences | {len(counts):,} types | vocab {len(vocab):,} "
          f"(min_freq={min_freq}, dropped {dropped:,} singletons)")
    (out_dir / "vocab.json").write_text(json.dumps(vocab), encoding="utf-8")

    # THE single shared base, drawn ONCE. Uniform(-0.25, 0.25) matches the usual
    # embedding-layer default.
    rng = np.random.default_rng(seed)
    base = rng.uniform(-0.25, 0.25, (len(vocab), dim)).astype(np.float32)
    base[index[PAD]] = 0.0

    np.save(out_dir / "E0_random.npy", base)
    print(f"  E0_random  {len(vocab):>6,} rows, 0 filled  (the floor)")

    for name, filename in FILES.items():
        key = name.split()[0]                      # "E1 GloVe (general)" -> "E1"
        kv = load(filename)
        matrix, hits = build_matrix(index, base, kv, skip=(PAD, UNK))
        np.save(out_dir / f"{key}.npy", matrix)
        print(f"  {key}         {len(vocab):>6,} rows, {hits:,} filled "
              f"({100.0 * hits / len(vocab):.1f}%)")
        del kv, matrix
        gc.collect()


if TRAIN:
    build_embedding_matrices()

The cell below verifies the two invariants the whole ablation rests on — same shape and
row order everywhere, and untouched rows identical across the four files.

In [7]:
import numpy as np
from src.vocab import load_vocab

MATRICES = ROOT / "models" / "emb_matrices"
vocab, index = load_vocab(MATRICES / "vocab.json")
E = {name: np.load(MATRICES / f"{name}.npy") for name in ["E0_random", "E1", "E2", "E3"]}

print(f"shared vocabulary: {len(vocab):,} words")
print(f"row 0 = {vocab[0]!r} (all zeros: {not E['E1'][0].any()}),  row 1 = {vocab[1]!r}\n")

rows = []
for name, matrix in E.items():
    filled = (matrix != E["E0_random"]).any(axis=1)      # rows the embedding overwrote
    rows.append({"matrix": f"{name}.npy", "shape": str(matrix.shape),
                 "rows filled": f"{int(filled.sum()):,}",
                 "coverage": f"{filled.mean():.1%}"})
display(pd.DataFrame(rows))

# The rows NO embedding covered must be identical in every file.
uncovered = ~((E["E1"] != E["E0_random"]).any(axis=1) | (E["E2"] != E["E0_random"]).any(axis=1))
print(f"\n{int(uncovered.sum()):,} rows covered by neither E1 nor E2")
print("identical across E0/E1/E2/E3:",
      np.array_equal(E["E0_random"][uncovered], E["E1"][uncovered])
      and np.array_equal(E["E1"][uncovered], E["E2"][uncovered]))

shared vocabulary: 12,220 words
row 0 = '<pad>' (all zeros: True),  row 1 = '<unk>'



,matrix,shape,rows filled,coverage
0,E0_random.npy,"(12220, 300)",0,0.0%
1,E1.npy,"(12220, 300)","9,413",77.0%
2,E2.npy,"(12220, 300)","11,160",91.3%
3,E3.npy,"(12220, 300)","12,218",100.0%



933 rows covered by neither E1 nor E2
identical across E0/E1/E2/E3: True


Four files, same shape, same word order, same untouched random rows. Loading a different
one is the *only* change between runs 3, 4, 5 and 6.

Note E3 fills 12,218 of 12,220 rows — everything except `<pad>` and `<unk>`. It is the only
matrix with no random rows at all, which is itself a difference between the runs and worth
mentioning when interpreting them.

---

## What this notebook produced

| Artefact | Contents |
|---|---|
| `models/w2v.kv`, `models/ft.kv` | E2 and E3, trained on 1.88M PubMed sentences |
| `models/glove.kv` | E1, downloaded |
| `models/emb_matrices/vocab.json` | the shared 12,220-word vocabulary |
| `models/emb_matrices/E{0,1,2,3}.npy` | four 12,220 x 300 matrices, identical but for the numbers |

**Next:** [4 — Stage 1](04_stage1_classification.ipynb), where those four matrices are
put into the same model and the headline result appears.